# Turtle Breakout 07 — what happens after a losing turtle trade, and does stop-and-reverse pay?

Pre-registration: `discussions/Turtle_Breakout/009_claude_tb07_preregistration.md`. Code: `src/data_engineering/modeling/turtle_reverse.py`.

Original daily turtle (TB 01 rules, long only). At every **losing stop exit** ($R<0$) a **short** is opened at the exit fill, with mirrored exits: fixed stop = price + 2 ATR, trailing = 10-day high, and closed at the next long breakout. Costs: taker 0.05% + 1 bp per side, funding.
**Decision (confirmation = BTC/ETH/SOL/XRP from 2020-10-15):** G1 mean $R_{rev}$ > 0 with month-cluster 95% lower bound > 0; G2 above random flat-day shorts (2,000 runs, empirical p < 0.05). Discovery = 26 other perps, same statistics, reported.

Environment: `TB01_DATA_DIR` (default `BINANCE_UM_DIR\long`), `TB07_SYNTHETIC=1`.

In [ ]:
import os, sys, json, time, gc, subprocess, tempfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 120)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_trade_features as tf, turtle_reverse as tr

SYNTHETIC = os.getenv('TB07_SYNTHETIC', '0') == '1'
P = tb.Params(); RP = tr.RParams()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb07-')) / '07'; RP = tr.RParams(n_null=500, n_boot=300)
    ASSETS = ['BTC', 'ETH', 'SOL', 'XRP', 'BNB', 'DOGE', 'ADA', 'LINK']
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '07'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
    ASSETS = list(tf.UNIVERSE)
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_reverse.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
print(json.dumps(tr.registration(RP), indent=1, default=str))

## 1. Long trades, reversals, random flat-day shorts and the event study, asset by asset

In [ ]:
TS, RS, POOLS, ES, COV = [], [], {}, [], []
for i, a in enumerate(ASSETS):
    t1 = time.time()
    if SYNTHETIC:
        b = tb.synthetic_5m(1800, seed=70 + i, trend=0.0, start='2020-01-01'); f = tb.synthetic_funding(b, 0.0001)
    else:
        pu = DATA / f'{a}USDT_um_1m.parquet'
        if not pu.exists():
            COV.append({'asset': a, 'status': 'no perp file'}); print(a, 'missing'); continue
        b, _ = tf.load_um(pu)
        fpth = DATA / 'basis' / 'funding' / f'{a}USDT_funding.parquet'
        f = tb.funding_events(pd.read_parquet(fpth)) if fpth.exists() else None
    d = tb.prepare_asset(a, b, f, P)
    conf = a in tf.CONFIRM_ASSETS
    start = tb.COMMON_START if conf else b.index[0] + pd.Timedelta(days=30)
    end = tr.TON_CUTOFF if a == 'TON' else None
    T = tf.run_trades(d, P, start, end)
    if not len(T):
        COV.append({'asset': a, 'status': 'no trades'}); continue
    k_lo = max(0, int(tb.period_id(pd.DatetimeIndex([start]), 'day')[0]) - d.pid0)
    k_hi = None if end is None else int(tb.period_id(pd.DatetimeIndex([end]), 'day')[0]) - d.pid0
    R = tr.reverse_trades(d, T, RP)
    if end is not None and len(R):
        R = R[R.rev_exit_time < end]
    pool = tr.random_short_pool(d, T, k_lo, RP, k_hi)
    E = tr.event_study(d, T, pool, RP)
    s = 'confirm' if conf else 'discovery'
    T = T.assign(set=s); R = R.assign(set=s); E = E.assign(set=s); pool = pool.assign(set=s)
    TS.append(T); RS.append(R); POOLS[a] = pool; ES.append(E)
    COV.append({'asset': a, 'status': 'ok', 'set': s, 'first_bar': b.index[0], 'last_bar': b.index[-1], 'longs': len(T),
                'losing_stop_exits': len(tr.losing_exits(T)), 'reversals': len(R), 'flat_days': len(pool),
                'sum_R_long': float(T.R.sum()), 'sum_R_rev': float(R.R_rev.sum()) if len(R) else 0.0, 'funding': f is not None})
    print(f"{a}: {len(T)} longs, {len(R)} reversals, {time.time() - t1:.0f}s")
    del b, d; gc.collect()
COV = pd.DataFrame(COV); display(COV)
TALL = pd.concat(TS, ignore_index=True); RALL = pd.concat(RS, ignore_index=True); EALL = pd.concat(ES, ignore_index=True)

## 2. Decision: stop-and-reverse on the confirmation set (binding), discovery reported

In [ ]:
RESULTS = {}
for s, seed in (('confirm', RP.seed), ('discovery', RP.seed + 1)):
    R = RALL[RALL.set == s]
    counts = R.groupby('asset').size().to_dict()
    nm = tr.null_means(POOLS, counts, RP, seed)
    RESULTS[s] = tr.evaluate(R, nm, RP, seed); RESULTS[s]['null_draws'] = nm
DEC = {s: {k: v for k, v in r.items() if k != 'null_draws'} for s, r in RESULTS.items()}
DEC['verdict'] = DEC['confirm']['verdict']
print(json.dumps(DEC, indent=1, default=float))

## 3. The event study: what price does after losing exits, winning exits and random flat days

In [ ]:
EV = pd.concat([tr.event_table(EALL[EALL.set == s], RP).assign(set=s) for s in ('confirm', 'discovery')], ignore_index=True)
display(EV.round(3))
print('mean_move_atr: forward move in ATR (negative = price fell). short_R_net: net R of a short opened at the start and held exactly h days.')

## 4. Breakdowns (reported)

In [ ]:
def summ(g):
    return pd.Series({'n': len(g), 'mean_R_rev': g.R_rev.mean(), 'median_R_rev': g.R_rev.median(), 'win_rate': (g.R_rev > 0).mean(),
                      'sum_R_rev': g.R_rev.sum(), 'median_hold_d': g.hold_days.median(), 'share_R_rev_ge_2': (g.R_rev >= 2).mean()})
BY_REASON = RALL.groupby(['set', 'long_reason']).apply(summ); display(BY_REASON.round(3))
BY_EXIT = RALL.groupby(['set', 'rev_reason']).apply(summ); display(BY_EXIT.round(3))
BY_YEAR = RALL.assign(year=pd.DatetimeIndex(RALL.rev_time).year).groupby(['set', 'year']).apply(summ); display(BY_YEAR.round(3))
BY_ASSET = RALL[RALL.set == 'confirm'].groupby('asset').apply(summ); display(BY_ASSET.round(3))
BY_LOSS = RALL.assign(long_R_bucket=pd.cut(RALL.long_R, [-10, -1.0, -0.75, -0.5, -0.25, 0])).groupby(['set', 'long_R_bucket'], observed=True).apply(summ)
display(BY_LOSS.round(3))
top = RALL[RALL.set == 'confirm'].sort_values('R_rev', ascending=False)
if len(top):
    print('confirmation: share of total reverse R from the top 3 trades =', round(top.R_rev.head(3).sum() / top.R_rev.sum(), 2) if top.R_rev.sum() != 0 else np.nan)

## 5. Curves: long only vs long + reverse (confirmation, 1R per trade, booked at exit)

In [ ]:
Tc = TALL[TALL.set == 'confirm']; Rc = RALL[RALL.set == 'confirm']
CUR = tr.curves(Tc, Rc)
d_long = CUR.long_only.diff().fillna(CUR.long_only.iloc[0]); d_both = CUR.long_plus_reverse.diff().fillna(CUR.long_plus_reverse.iloc[0])
sh = lambda x: float(x.mean() / x.std(ddof=1) * np.sqrt(365)) if x.std() > 0 else np.nan
CSUM = pd.DataFrame({'total_R': CUR.iloc[-1], 'max_dd_R': [tr.max_dd(CUR[c]) for c in CUR],
                     'sharpe_daily_R': [sh(CUR[c].diff().fillna(CUR[c].iloc[0])) for c in CUR]})
display(CSUM.round(3))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for c, col in zip(CUR, ('#1f5aa6', '#b33f40', '#3a7d44')):
    axes[0].plot(CUR.index, CUR[c], label=c, color=col, lw=1.4)
axes[0].axhline(0, color='k', lw=0.6); axes[0].set_ylabel('cumulative R'); axes[0].legend(frameon=False); axes[0].grid(alpha=0.25)
axes[0].set_title('Confirmation (BTC/ETH/SOL/XRP): unit risk per trade', loc='left', fontsize=10)
for s, col in (('confirm', '#1f5aa6'), ('discovery', '0.55')):
    nm = RESULTS[s]['null_draws']
    axes[1].hist(nm, bins=50, alpha=0.55, color=col, density=True, label=f'{s}: random flat-day shorts')
    axes[1].axvline(RESULTS[s].get('mean_R_rev', np.nan), color=col, lw=2, ls='--')
axes[1].set_xlabel('mean R per short'); axes[1].legend(frameon=False, fontsize=8); axes[1].set_title('Real reversals (dashed) vs null', loc='left', fontsize=10)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
for s, mk in (('confirm', 'o'), ('discovery', 's')):
    for kind, col in (('after_loss', '#b33f40'), ('after_win', '#3a7d44'), ('random_flat', '0.5')):
        g = EV[(EV.set == s) & (EV.kind == kind)]
        ax.plot(g.h_days, g.mean_move_atr, marker=mk, color=col, ls='-' if s == 'confirm' else ':', label=f'{s} {kind}')
ax.axhline(0, color='k', lw=0.6); ax.set_xlabel('days after the exit'); ax.set_ylabel('mean forward move (ATR)'); ax.legend(fontsize=7, frameon=False, ncol=2); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 6. Save

In [ ]:
import platform
DEC_OUT = {**DEC, 'registration': tr.registration(RP), 'synthetic': SYNTHETIC}
(OUT / '07_decision.json').write_text(json.dumps(DEC_OUT, indent=1, default=str))
COV.to_csv(OUT / '07_coverage.csv', index=False); EV.to_csv(OUT / '07_event_study.csv', index=False)
RALL.to_csv(OUT / '07_reversals.csv', index=False)
pd.concat([BY_REASON.assign(table='long_reason'), BY_EXIT.assign(table='rev_reason')]).to_csv(OUT / '07_by_reason.csv')
BY_YEAR.to_csv(OUT / '07_by_year.csv'); BY_ASSET.to_csv(OUT / '07_by_asset.csv'); BY_LOSS.to_csv(OUT / '07_by_long_R.csv')
CSUM.to_csv(OUT / '07_curves_summary.csv'); CUR.to_csv(OUT / '07_curves_daily.csv')
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '07_manifest.json').write_text(json.dumps({'version': tr.VERSION, 'git': git, 'python': platform.python_version(),
    'created_utc': pd.Timestamp.now('UTC').isoformat(), 'assets': ASSETS, 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print('VERDICT:', DEC['verdict'])